# Detecção e Reconstrução de Malha Viária a partir de Imagens de Satélite

**Disciplina:** Reconhecimento de Padrões  
**Integrantes:** Beatriz Vocurca Frade, Johnatan Augusto Moreira do Carmo e Marina Alves Resende

Este notebook implementa um pipeline de visão computacional clássica para detectar vias, gerar máscaras segmentadas, sobrepor o resultado na imagem original e extrair um grafo da malha viária.

## 0. Setup

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR     = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
GRAPHS_DIR  = PROJECT_ROOT / "data" / "graphs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raiz: {PROJECT_ROOT}")


Raiz: /content


## 1. Pipeline de detecção

In [ ]:
from __future__ import annotations

import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import cv2
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from skimage.morphology import skeletonize

SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


@dataclass
class RoadDetectionResult:
    image_name: str
    rgb: np.ndarray
    road_mask: np.ndarray
    class_map: np.ndarray
    skeleton: np.ndarray
    graph: nx.Graph
    segmented: np.ndarray
    overlay: np.ndarray
    graph_overlay: np.ndarray
    metrics: dict


def list_images(raw_dir: Path) -> list[Path]:
    raw_dir.mkdir(parents=True, exist_ok=True)
    return sorted(
        p for p in raw_dir.iterdir()
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    )


def load_rgb_image(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Nao foi possivel carregar: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def build_synthetic_demo(size: int = 768) -> np.ndarray:
    rng = np.random.default_rng(7)
    base = np.zeros((size, size, 3), dtype=np.uint8)
    base[..., 0] = rng.normal(82, 14, (size, size)).clip(0, 255)
    base[..., 1] = rng.normal(115, 18, (size, size)).clip(0, 255)
    base[..., 2] = rng.normal(82, 14, (size, size)).clip(0, 255)
    asphalt, dirt = (92, 92, 88), (154, 116, 72)
    roads = [
        ((60, 140), (710, 170), 28, asphalt),
        ((120, 90), (520, 690), 22, asphalt),
        ((60, 520), (720, 460), 24, dirt),
        ((420, 40), (460, 720), 18, dirt),
        ((120, 650), (660, 260), 16, asphalt),
    ]
    for p1, p2, width, color in roads:
        cv2.line(base, p1, p2, color, width, lineType=cv2.LINE_AA)
    for _ in range(45):
        x, y = rng.integers(0, size - 60, 2)
        w, h = rng.integers(18, 54, 2)
        color = tuple(int(v) for v in rng.normal([135, 128, 120], [25, 18, 18]))
        cv2.rectangle(base, (int(x), int(y)), (int(x + w), int(y + h)), color, -1)
    return base


def resize_if_needed(rgb: np.ndarray, max_side: int = 1400) -> tuple[np.ndarray, float]:
    h, w = rgb.shape[:2]
    scale = min(1.0, max_side / max(h, w))
    if scale >= 1.0:
        return rgb.copy(), 1.0
    new_size = (int(round(w * scale)), int(round(h * scale)))
    return cv2.resize(rgb, new_size, interpolation=cv2.INTER_AREA), scale


def _odd_kernel(value: int, minimum: int = 3) -> int:
    value = max(minimum, int(value))
    return value if value % 2 == 1 else value + 1


def _local_std(gray: np.ndarray, kernel_size: int) -> np.ndarray:
    gray_f = gray.astype(np.float32)
    mean = cv2.blur(gray_f, (kernel_size, kernel_size))
    mean_sq = cv2.blur(gray_f * gray_f, (kernel_size, kernel_size))
    return np.sqrt(np.maximum(mean_sq - mean * mean, 0))


def preprocess_image(rgb: np.ndarray, max_side: int = 1400) -> dict:
    resized, scale = resize_if_needed(rgb, max_side=max_side)
    denoised = cv2.bilateralFilter(resized, d=7, sigmaColor=45, sigmaSpace=45)
    lab = cv2.cvtColor(denoised, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_equalized = clahe.apply(l_channel)
    enhanced = cv2.cvtColor(cv2.merge([l_equalized, a_channel, b_channel]), cv2.COLOR_LAB2RGB)
    gray = cv2.cvtColor(enhanced, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(enhanced, cv2.COLOR_RGB2HSV)
    h, w = gray.shape
    local_kernel = _odd_kernel(min(h, w) // 45, minimum=9)
    return {
        "rgb": resized, "enhanced": enhanced, "gray": gray, "hsv": hsv,
        "lab": cv2.merge([l_equalized, a_channel, b_channel]),
        "local_std": _local_std(gray, local_kernel),
        "scale": scale, "local_kernel": local_kernel,
    }


def compute_line_support(gray: np.ndarray) -> np.ndarray:
    """Mapa float32 [0,1]: alto onde ha estruturas lineares (vias).

    Combina dilatacao direcional de bordas Canny (6 orientacoes) com
    detector de cristas via Hessiana multi-escala (Frangi simplificado).
    Solo/campo aberto = textura homogenea sem bordas lineares -> valor baixo.
    Estrada de terra = cristas com bordas paralelas -> valor alto.
    """
    h, w = gray.shape
    min_side = min(h, w)
    line_len = max(11, min_side // 55)

    blur = cv2.GaussianBlur(gray, (5, 5), 1.5)
    edges = cv2.Canny(blur, 20, 80)

    # Dilatacao direcional em 6 orientacoes
    dir_acc = np.zeros((h, w), dtype=np.float32)
    for ang_deg in range(0, 180, 30):
        ang = np.deg2rad(ang_deg)
        dx = int(round(np.cos(ang) * line_len))
        dy = int(round(np.sin(ang) * line_len))
        half = max(abs(dx), abs(dy), 1)
        ks = half * 2 + 1
        k = np.zeros((ks, ks), dtype=np.uint8)
        c = half
        cv2.line(k, (c - dx, c - dy), (c + dx, c + dy), 1, 1)
        dir_acc += cv2.dilate(edges, k).astype(np.float32)

    # Hessiana multi-escala (Frangi simplificado, sem dependencia extra)
    ridge = np.zeros((h, w), dtype=np.float32)
    for sigma in (2.0, 4.5):
        b = cv2.GaussianBlur(gray.astype(np.float32), (0, 0), sigma)
        Ixx = cv2.Sobel(b, cv2.CV_32F, 2, 0, ksize=3)
        Iyy = cv2.Sobel(b, cv2.CV_32F, 0, 2, ksize=3)
        Ixy = cv2.Sobel(b, cv2.CV_32F, 1, 1, ksize=3)
        tr = (Ixx + Iyy) / 2
        disc = np.sqrt(((Ixx - Iyy) / 2) ** 2 + Ixy ** 2)
        lam1 = tr + disc
        lam2 = tr - disc
        S = np.sqrt(lam1 ** 2 + lam2 ** 2)
        lmax = np.maximum(np.abs(lam1), np.abs(lam2))
        lmin = np.minimum(np.abs(lam1), np.abs(lam2))
        Rb = lmin / (lmax + 1e-9)
        ridge = np.maximum(ridge, S / (S.max() + 1e-9) * np.exp(-(Rb ** 2) / 0.5))

    d_norm = dir_acc / (dir_acc.max() + 1e-9)
    r_norm = ridge / (ridge.max() + 1e-9)
    combined = np.maximum(d_norm * 0.7, r_norm * 0.6)
    return np.clip(combined / (combined.max() + 1e-9), 0.0, 1.0).astype(np.float32)


def segment_roads(pre: dict) -> np.ndarray:
    """Segmentacao com regra: candidate = asphalt_like | (dirt_like & line_support).

    dirt_like sozinho nao basta: exige suporte linear para evitar que
    solo exposto e campo aberto sejam classificados como estrada de terra.
    """
    rgb       = pre["rgb"]
    gray      = pre["gray"]
    hsv       = pre["hsv"]
    local_std = pre["local_std"]

    h_ch, s_ch, v_ch = cv2.split(hsv)

    r = rgb[..., 0].astype(np.int16)
    g = rgb[..., 1].astype(np.int16)
    b = rgb[..., 2].astype(np.int16)

    texture_limit = np.percentile(local_std, 55)
    homogeneous   = local_std <= max(12, texture_limit)
    low_sat       = s_ch <= max(42, np.percentile(s_ch, 45))
    visible       = v_ch >= 35

    spread  = (np.maximum.reduce([r, g, b]) - np.minimum.reduce([r, g, b])).astype(np.int16)
    grayish = spread <= 38

    dirt_hue  = ((h_ch <= 28) | (h_ch >= 165)) & (s_ch >= 20)
    dirt_rgb  = (r > g - 10) & (g >= b - 14) & ((r - b) >= 15)
    dirt_like = dirt_hue & dirt_rgb & homogeneous & visible

    asphalt_like = low_sat & grayish & homogeneous & visible

    excess_g   = (2 * g - r - b).astype(np.int16)
    vegetation = (excess_g > np.percentile(excess_g, 70)) & (g > r + 6)
    shadow     = (v_ch < 38) & (s_ch < 50)

    line_mask = compute_line_support(gray) > 0.15

    candidate  = asphalt_like | (dirt_like & line_mask)
    candidate &= ~vegetation
    candidate &= ~shadow

    return _postprocess_mask(candidate)


def _postprocess_mask(mask: np.ndarray) -> np.ndarray:
    mask_u8  = mask.astype(np.uint8) * 255
    H, W     = mask.shape
    min_side = min(H, W)

    ck = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd_kernel(min_side // 120, 5),) * 2)
    ok = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd_kernel(min_side // 300, 3),) * 2)
    clean = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, ck, iterations=2)
    clean = cv2.morphologyEx(clean,   cv2.MORPH_OPEN,  ok, iterations=1)

    total    = H * W
    min_area = max(80, int(total * 0.00025))
    max_blob = int(total * 0.05)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(clean, connectivity=8)
    keep = np.zeros(mask.shape, dtype=bool)

    for lbl in range(1, n):
        area  = stats[lbl, cv2.CC_STAT_AREA]
        bw    = stats[lbl, cv2.CC_STAT_WIDTH]
        bh    = stats[lbl, cv2.CC_STAT_HEIGHT]
        elong = max(bw, bh) / max(1, min(bw, bh))
        fill  = area / max(1, bw * bh)

        if area < min_area:
            continue
        if area > max_blob and elong < 2.5:
            continue
        if area > max_blob * 0.4 and fill > 0.65 and elong < 2.5:
            continue
        keep |= labels == lbl

    return keep


def classify_road_type(rgb: np.ndarray, road_mask: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)
    r = rgb[..., 0].astype(np.int16)
    g = rgb[..., 1].astype(np.int16)
    b = rgb[..., 2].astype(np.int16)
    spread = np.maximum.reduce([r, g, b]) - np.minimum.reduce([r, g, b])
    dirt    = road_mask & (((h <= 30) | (h >= 165)) & (s >= 30)) & (r >= g - 10) & (g >= b - 18)
    asphalt = road_mask & ((s < 60) | (spread < 35)) & ~dirt
    labels  = np.zeros(road_mask.shape, dtype=np.uint8)
    labels[asphalt] = 1
    labels[dirt]    = 2
    remaining = road_mask & (labels == 0)
    labels[remaining & (s <  75)] = 1
    labels[remaining & (s >= 75)] = 2
    return labels


def make_segmented_image(class_map: np.ndarray) -> np.ndarray:
    seg = np.zeros((*class_map.shape, 3), dtype=np.uint8)
    seg[class_map == 1] = (40, 160, 255)
    seg[class_map == 2] = (220, 145, 40)
    return seg


def make_overlay(rgb: np.ndarray, class_map: np.ndarray, alpha: float = 0.48) -> np.ndarray:
    seg = make_segmented_image(class_map)
    overlay = rgb.copy()
    road_pixels = class_map > 0
    overlay[road_pixels] = (
        (1 - alpha) * overlay[road_pixels].astype(np.float32)
        + alpha * seg[road_pixels].astype(np.float32)
    ).astype(np.uint8)
    return overlay


def compute_skeleton(road_mask: np.ndarray) -> np.ndarray:
    return skeletonize(road_mask > 0)


def _neighbors_8(point, shape):
    y, x = point
    height, width = shape
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if dy == 0 and dx == 0:
                continue
            yy, xx = y + dy, x + dx
            if 0 <= yy < height and 0 <= xx < width:
                yield yy, xx


def _pixel_edge(a, b):
    return tuple(sorted((a, b)))


def _skeleton_degree(skeleton: np.ndarray) -> np.ndarray:
    kernel = np.ones((3, 3), dtype=np.uint8)
    counts = cv2.filter2D(skeleton.astype(np.uint8), -1, kernel, borderType=cv2.BORDER_CONSTANT)
    return counts - skeleton.astype(np.uint8)


def skeleton_to_graph(skeleton: np.ndarray, min_edge_length: int = 4) -> nx.Graph:
    skeleton = skeleton.astype(bool)
    degree = _skeleton_degree(skeleton)
    node_mask = skeleton & ((degree == 1) | (degree >= 3))
    num_labels, labels = cv2.connectedComponents(node_mask.astype(np.uint8), connectivity=8)
    graph = nx.Graph()
    if num_labels <= 1:
        return graph
    node_label_to_id: dict[int, str] = {}
    for comp in range(1, num_labels):
        ys, xs = np.where(labels == comp)
        if len(xs) == 0:
            continue
        node_id = str(comp - 1)
        node_type = "intersecao" if int(degree[ys, xs].max()) >= 3 else "extremidade"
        node_label_to_id[comp] = node_id
        graph.add_node(node_id, x=float(xs.mean()), y=float(ys.mean()), tipo=node_type, pixels=int(len(xs)))
    visited: set = set()
    shape = skeleton.shape
    for comp, src_id in node_label_to_id.items():
        for src_px in zip(*np.where(labels == comp)):
            for nb in _neighbors_8(src_px, shape):
                if not skeleton[nb] or labels[nb] == comp:
                    continue
                first_edge = _pixel_edge(src_px, nb)
                if first_edge in visited:
                    continue
                path = [src_px, nb]
                prev, curr = src_px, nb
                visited.add(first_edge)
                while True:
                    curr_label = int(labels[curr])
                    if curr_label > 0 and curr_label != comp:
                        tgt_id = node_label_to_id.get(curr_label)
                        if tgt_id and tgt_id != src_id and len(path) >= min_edge_length:
                            pts = [(int(x), int(y)) for y, x in path]
                            length = sum(
                                math.hypot(x2 - x1, y2 - y1)
                                for (x1, y1), (x2, y2) in zip(pts, pts[1:])
                            )
                            geo = " ".join(f"{x},{y}" for x, y in pts)
                            if graph.has_edge(src_id, tgt_id):
                                if length < float(graph[src_id][tgt_id].get("length_px", float("inf"))):
                                    graph[src_id][tgt_id].update(length_px=length, points_count=len(pts), geometry=geo)
                            else:
                                graph.add_edge(src_id, tgt_id, length_px=length, points_count=len(pts), geometry=geo)
                        break
                    nexts = [p for p in _neighbors_8(curr, shape) if skeleton[p] and p != prev]
                    if not nexts:
                        break
                    nxt = next((p for p in nexts if _pixel_edge(curr, p) not in visited), None)
                    if nxt is None:
                        break
                    visited.add(_pixel_edge(curr, nxt))
                    prev, curr = curr, nxt
                    path.append(curr)
    return graph


def largest_connected_subgraph(graph: nx.Graph) -> nx.Graph:
    if graph.number_of_nodes() == 0 or nx.is_connected(graph):
        return graph.copy()
    return graph.subgraph(max(nx.connected_components(graph), key=len)).copy()


def _parse_geometry(geometry: str) -> list[tuple[int, int]]:
    points = []
    for pair in geometry.split():
        try:
            x_str, y_str = pair.split(",")
            points.append((int(x_str), int(y_str)))
        except ValueError:
            continue
    return points


def draw_graph_overlay(rgb: np.ndarray, graph: nx.Graph) -> np.ndarray:
    canvas = rgb.copy()
    for _, _, data in graph.edges(data=True):
        pts = _parse_geometry(data.get("geometry", ""))
        if len(pts) >= 2:
            cv2.polylines(canvas, [np.array(pts, dtype=np.int32).reshape((-1, 1, 2))],
                          isClosed=False, color=(255, 40, 40), thickness=3)
    for _, data in graph.nodes(data=True):
        x, y = int(round(float(data.get("x", 0)))), int(round(float(data.get("y", 0))))
        color = (255, 255, 30) if data.get("tipo") == "intersecao" else (60, 255, 255)
        cv2.circle(canvas, (x, y), 6, color, -1, lineType=cv2.LINE_AA)
        cv2.circle(canvas, (x, y), 8, (0, 0, 0), 1, lineType=cv2.LINE_AA)
    return canvas


def save_outputs(result: RoadDetectionResult, results_dir: Path, graphs_dir: Path) -> None:
    results_dir.mkdir(parents=True, exist_ok=True)
    graphs_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(result.image_name).stem
    cv2.imwrite(str(results_dir / f"{stem}_segmentada.png"),   cv2.cvtColor(result.segmented, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(results_dir / f"{stem}_sobreposicao.png"), cv2.cvtColor(result.overlay, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(results_dir / f"{stem}_grafo.png"),        cv2.cvtColor(result.graph_overlay, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(results_dir / f"{stem}_mascara.png"),      result.road_mask.astype(np.uint8) * 255)
    cv2.imwrite(str(results_dir / f"{stem}_esqueleto.png"),    result.skeleton.astype(np.uint8) * 255)
    nodes = [{"id": n, "x": float(d.get("x", 0)), "y": float(d.get("y", 0)),
              "tipo": d.get("tipo"), "pixels": int(d.get("pixels", 0))}
             for n, d in result.graph.nodes(data=True)]
    edges = [{"source": s, "target": t, "length_px": float(d.get("length_px", 0)),
              "points": [[x, y] for x, y in _parse_geometry(d.get("geometry", ""))]}
             for s, t, d in result.graph.edges(data=True)]
    with (graphs_dir / f"{stem}_grafo.json").open("w", encoding="utf-8") as f:
        json.dump({"image": result.image_name, "metrics": result.metrics,
                   "nodes": nodes, "edges": edges}, f, indent=2)
    nx.write_graphml(result.graph, graphs_dir / f"{stem}_grafo.graphml")


def process_rgb_image(image_name: str, rgb: np.ndarray, max_side: int = 1400) -> RoadDetectionResult:
    pre           = preprocess_image(rgb, max_side=max_side)
    road_mask     = segment_roads(pre)
    class_map     = classify_road_type(pre["rgb"], road_mask)
    skeleton      = compute_skeleton(road_mask)
    full_graph    = skeleton_to_graph(skeleton)
    graph         = largest_connected_subgraph(full_graph)
    segmented     = make_segmented_image(class_map)
    overlay       = make_overlay(pre["rgb"], class_map)
    graph_overlay = draw_graph_overlay(overlay, graph)
    road_pixels   = int(road_mask.sum())
    n_comp        = int(cv2.connectedComponents(road_mask.astype(np.uint8), connectivity=8)[0]) - 1
    metrics = {
        "scale":                     float(pre["scale"]),
        "road_pixel_ratio":          float(road_pixels / max(1, road_mask.size)),
        "road_pixels":               road_pixels,
        "mask_components":           n_comp,
        "nodes_total":               int(graph.number_of_nodes()),
        "edges_total":               int(graph.number_of_edges()),
        "components_before_largest": int(nx.number_connected_components(full_graph)) if full_graph.number_of_nodes() else 0,
        "asphalt_pixels":            int((class_map == 1).sum()),
        "dirt_pixels":               int((class_map == 2).sum()),
    }
    return RoadDetectionResult(image_name=image_name, rgb=pre["rgb"], road_mask=road_mask,
                               class_map=class_map, skeleton=skeleton, graph=graph,
                               segmented=segmented, overlay=overlay,
                               graph_overlay=graph_overlay, metrics=metrics)


def process_dataset(raw_dir: Path, results_dir: Path, graphs_dir: Path,
                    max_side: int = 1400, use_synthetic_if_empty: bool = True
                    ) -> tuple[list[RoadDetectionResult], pd.DataFrame]:
    image_paths = list_images(raw_dir)
    results: list[RoadDetectionResult] = []
    if not image_paths and use_synthetic_if_empty:
        result = process_rgb_image("demo_sintetico.png", build_synthetic_demo(), max_side)
        save_outputs(result, results_dir, graphs_dir)
        results.append(result)
    else:
        for p in image_paths:
            result = process_rgb_image(p.name, load_rgb_image(p), max_side)
            save_outputs(result, results_dir, graphs_dir)
            results.append(result)
    summary = pd.DataFrame([{"image": r.image_name, **r.metrics} for r in results])
    return results, summary


def plot_all_results(results: list[RoadDetectionResult]) -> None:
    for result in results:
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        views = [
            ("Imagem original",        result.rgb,           None),
            ("Mascara de vias",        result.road_mask,     "gray"),
            ("Classes: asfalto/terra", result.segmented,     None),
            ("Sobreposicao",           result.overlay,       None),
            ("Esqueleto",              result.skeleton,      "gray"),
            ("Grafo extraido",         result.graph_overlay, None),
        ]
        for ax, (title, img, cmap) in zip(axes.ravel(), views):
            ax.imshow(img, cmap=cmap)
            ax.set_title(title)
            ax.axis("off")
        fig.suptitle(result.image_name)
        fig.tight_layout()

print("Pipeline carregado.")
print(f"Imagens encontradas: {len(list_images(RAW_DIR))}")


## 2. Como usar

1. Coloque imagens de satélite em `data/raw/`.
2. Execute todas as células deste notebook.
3. As imagens de saída serão salvas em `data/results/`.
4. Os grafos serão exportados em `data/graphs/`, nos formatos JSON e GraphML.

Quando `data/raw/` estiver vazia, o notebook usa uma imagem sintética apenas para demonstrar que o pipeline está funcionando.


## 3. Ideia da solução

A segmentação combina normalização, CLAHE, suavização bilateral, descritores simples de cor, textura local, bordas e morfologia matemática. Depois, a máscara é esqueletonizada para extrair a topologia: pixels com grau 1 viram extremidades, pixels com grau maior ou igual a 3 viram interseções, e os caminhos entre esses vértices viram arestas do grafo.


## 4. Processamento em lote

A célula abaixo processa todas as imagens encontradas em `data/raw/`. O parâmetro `max_side` limita o maior lado da imagem para reduzir custo computacional, mantendo a proporção original.


In [ ]:
results, summary = process_dataset(
    raw_dir=RAW_DIR,
    results_dir=RESULTS_DIR,
    graphs_dir=GRAPHS_DIR,
    max_side=1400,
    use_synthetic_if_empty=True,
)

summary

## 5. Visualização dos resultados

Para cada imagem, são exibidos: imagem original, máscara binária, classes de via, sobreposição, esqueleto e grafo extraído.


In [ ]:
plot_all_results(results)

## 6. Arquivos gerados

Para cada imagem processada, o pipeline salva:

- `*_segmentada.png`: vias coloridas por classe;
- `*_sobreposicao.png`: vias sobrepostas à imagem original;
- `*_grafo.png`: grafo desenhado sobre a sobreposição;
- `*_mascara.png`: máscara binária;
- `*_esqueleto.png`: esqueleto da máscara;
- `*_grafo.json`: nós, arestas e pontos de cada segmento;
- `*_grafo.graphml`: grafo para ferramentas como Gephi, Cytoscape ou NetworkX.


In [ ]:
print("Resultados em:", RESULTS_DIR)
for p in sorted(RESULTS_DIR.glob("*")):
    if p.is_file() and p.name != ".gitkeep":
        print(" -", p.name)
print("Grafos em:", GRAPHS_DIR)
for p in sorted(GRAPHS_DIR.glob("*")):
    if p.is_file() and p.name != ".gitkeep":
        print(" -", p.name)


## 7. Interpretação e limitações

A solução é propositalmente explicável. Ela tende a funcionar melhor quando as vias têm textura homogênea e contraste razoável em relação ao entorno. Os principais erros esperados são falsos positivos em telhados claros, solo exposto, sombras lineares e regiões agrícolas; falsos negativos podem ocorrer em vias muito estreitas, arborizadas ou com pavimento visualmente parecido com o ambiente.
